# Latin Hypercube Sampling
example code to sample parameter space with LHC

In [2]:
from scipy.stats import qmc
import scipy
import pandas as pd
import numpy as np
from datetime import datetime
import tomllib

In [3]:
def latin_hypercube_sampling(param_dict=None, 
                             n_samples=100, 
                             scramble=True,
                             optimization='random-cd',
                             strength=1,
                             random_seed=None):
    """Latin HyperCube sampler
    
    Parameters
    ----------
    param_dict [dict]: dictionary of params with bounds
                "param1": [lower_bound, upper_bound]
    n_samples [int]: number of samples to generate
    
    Returns
    ----------
    np.array: array of samples
    
    """
    # lower and upper bounds of parameters
    lower_bounds = [value[0] for value in param_dict.values()]
    upper_bounds = [value[1] for value in param_dict.values()]

    # random seed between 0-1_000_000, saved for reproducibility
    seed = np.random.randint(0, 10**6 + 1) if random_seed is None else random_seed

    # latin hypercube sampler
    sampler = qmc.LatinHypercube(d=len(lower_bounds),   
                                 scramble=scramble,
                                 optimization=optimization,
                                 strength=strength, # 1 produces a plain LHS, 2 produces an orthogonal array based LHS
                                 seed=seed)
    
    sample = sampler.random(n=n_samples)
    sample_array = qmc.scale(sample, lower_bounds, upper_bounds)
    
    dataframe = pd.DataFrame(data={key:sample_array[:,ind] for ind,key in enumerate(param_dict.keys())})
    
    return dataframe

In [4]:
# 1. read parameters of parameters from `parameters.toml`
with open("parameters.toml", "rb") as f:
    data = tomllib.load(f)

param_dict = {key: value["bounds"] for key, value in data.items()}

# 2. sample parameter space with LHC
df = latin_hypercube_sampling(param_dict=param_dict)

In [5]:
df.head()

,diatom_maximum_growth_rate,diatom_grazing_rate,diatom_mortality_rate,diatom_Ks_Si,diatom_Ks_N,diatom_Ks_P,diatom_Ks_Fe
0,1.044081,0.450076,0.214061,6.688786,0.336364,0.227935,0.364081
1,2.203304,0.787508,0.062952,8.481903,2.529926,0.341657,0.446456
2,3.255963,1.244960,0.184703,0.775422,3.279244,0.177794,0.433335
3,3.061781,0.877551,0.085882,1.421886,0.795402,0.159719,0.266968
4,2.450476,1.621101,0.008008,1.606844,2.654427,0.061956,0.046520
